In [4]:
import json
import os

In [24]:
class ChatOrchestrator:
    def __init__(
        self,
        client,
        model_name: str,
        system_prompt: str,
        tools: list = None,
        history_file: str = "history.json",
    ):
        self.client = client
        self.model_name = model_name
        self.system_prompt = system_prompt
        self.tools = tools or TOOLS_SCHEMA
        self.history_file = history_file
        self.history = self._load_history()

    # ── persistence ──────────────────────────────────────────────────────────

    def _load_history(self) -> list:
        if os.path.exists(self.history_file):
            try:
                with open(self.history_file, "r", encoding="utf-8") as f:
                    return json.load(f)
            except json.JSONDecodeError:
                return []
        return []

    def _save_history(self):
        with open(self.history_file, "w", encoding="utf-8") as f:
            json.dump(self.history, f, indent=4, ensure_ascii=False)

    def resetHistory(self):
        self.history = []
        if os.path.exists(self.history_file):
            os.remove(self.history_file)
        print(f"History file deleted: {self.history_file}")

    # ── agentic loop ─────────────────────────────────────────────────────────
    def chat(self, user_message: str) -> str:
        """
        Send a message and run the tool-call loop until the model
        produces a plain text reply, then persist the full exchange.
        """
        # ── /reset command ───────────────────────────────────────────────
        if user_message.strip().lower() == "/reset":
            self.resetHistory()
            return "Conversation history has been cleared. Starting fresh!"

        working_messages = (
            [{"role": "system", "content": self.system_prompt}]
            + self.history
            + [{"role": "user", "content": user_message}]
        )

        while True:
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=working_messages,
                tools=self.tools,
                tool_choice="auto",
            )

            choice = response.choices[0]
            assistant_msg = choice.message

            # ── No tool calls → final answer ─────────────────────────────────
            if not assistant_msg.tool_calls:
                final_text = assistant_msg.content

                self.history.append({"role": "user", "content": user_message})
                self.history.append({"role": "assistant", "content": final_text})
                self._save_history()

                return final_text

            # ── Tool calls requested: execute ALL of them ─────────────────────

            # Add the assistant's full message (lists all calls)
            working_messages.append(assistant_msg)

            # Execute every tool call and append ALL results before looping.
            # OpenAI requires a tool-response message for every tool_call_id
            # present in the preceding assistant message.
            for tool_call in assistant_msg.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)

                print(f"  [tool] {tool_name}({tool_args})")  # optional debug log

                try:
                    if tool_name in TOOL_FUNCTIONS:
                        tool_result = TOOL_FUNCTIONS[tool_name](tool_args)
                    else:
                        tool_result = f"Unknown tool '{tool_name}'."
                except Exception as e:
                    tool_result = f"Error calling {tool_name}: {type(e).__name__}: {e}"

                working_messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result),
                })

            # Loop → model sees ALL results and decides its next step

Weather

In [ ]:
import requests

def getWeather(city: str) -> float:
    """
    Retrieve the current temperature for a given city.

    Uses the Open-Meteo geocoding API to resolve the city name to coordinates,
    then queries the Open-Meteo weather API for the current temperature.

    Args:
        city (str): The name of the city to get the weather for.
                    Examples: "Haifa", "New York City", "Tokyo", "London".

    Returns:
        float: The current temperature in Celsius at the given city's location.

    Raises:
        ValueError: If the city name cannot be resolved to coordinates.
        requests.exceptions.HTTPError: If the API request returns an unsuccessful status code.
        requests.exceptions.RequestException: For underlying network or connection issues.

    Example:
        >>> temp = getWeather("Haifa")
        >>> print(f"Current temperature in Haifa: {temp}°C")
        Current temperature in Haifa: 24.1°C
    """
    with requests.Session() as session:
        # Step 1: Geocode the city name to lat/lon
        geo_response = session.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1, "language": "en", "format": "json"},
            timeout=30
        )
        geo_response.raise_for_status()
        geo_data = geo_response.json()

        if not geo_data.get("results"):
            raise ValueError(f"Could not find coordinates for city: '{city}'")

        lat = geo_data["results"][0]["latitude"]
        lon = geo_data["results"][0]["longitude"]

        # Step 2: Fetch current temperature from weather API
        weather_response = session.get(
            "https://api.open-meteo.com/v1/forecast",
            params={"latitude": lat, "longitude": lon, "current": "temperature_2m"},
            timeout=10
        )
        weather_response.raise_for_status()
        weather_data = weather_response.json()

    return float(weather_data["current"]["temperature_2m"])


# --- Usage ---
if __name__ == "__main__":
    for city in ["Haifa", "New York City"]:
        temp = getWeather(city)
        print(f"{city}: {temp}°C")

Haifa: 19.4°C
New York City: 5.9°C


Calculator

In [7]:
import ast
import operator
from typing import Union


def calculateMath(expression: str) -> float:
    """
    Safely evaluates a mathematical expression string using AST parsing.

    Uses Python's Abstract Syntax Tree to evaluate only explicit mathematical
    operations, avoiding the severe security risks of built-in eval().

    Args:
        expression (str): A mathematical expression, e.g. "50 * 3 / 2".

    Returns:
        float: The numeric result of the expression.

    Raises:
        TypeError:       If expression is not a string.
        ValueError:      If expression is empty or contains unsupported syntax.
        ZeroDivisionError: If the expression divides by zero.
        SyntaxError:     If the expression cannot be parsed.

    Examples:
        >>> calculateMath("50 * 3 / 2")
        75.0
        >>> calculateMath("10 + 2 ** 3")
        18.0
        >>> calculateMath("-(4 + 5) * 2")
        -18.0
        >>> calculateMath("100 % 3")
        1.0
        >>> calculateMath("10 // 3")
        3.0
    """

    if not isinstance(expression, str):
        raise TypeError(
            f"Expected a string expression, got {type(expression).__name__}"
        )
    expression = expression.strip()
    if not expression:
        raise ValueError("Expression must not be empty.")

    allowed_operators = {
        ast.Add:      operator.add,
        ast.Sub:      operator.sub,
        ast.Mult:     operator.mul,
        ast.Div:      operator.truediv,
        ast.FloorDiv: operator.floordiv,
        ast.Pow:      operator.pow,
        ast.Mod:      operator.mod,
        ast.USub:     operator.neg,
        ast.UAdd:     operator.pos,
    }

    def _evaluate(node: ast.AST) -> float:
        """Recursively walks the AST to evaluate mathematical nodes."""

        if isinstance(node, ast.Constant):
            if not isinstance(node.value, (int, float)):
                raise TypeError(
                    f"Only numeric constants are allowed, got {type(node.value).__name__}"
                )
            return float(node.value)

        elif isinstance(node, ast.BinOp):
            op_type = type(node.op)
            if op_type not in allowed_operators:
                raise TypeError(
                    f"Unsupported binary operator: {op_type.__name__}"
                )
            left  = _evaluate(node.left)
            right = _evaluate(node.right)
            # Explicit zero-division guard
            if op_type is ast.Div and right == 0:
                raise ZeroDivisionError("Division by zero is not allowed.")
            return allowed_operators[op_type](left, right)

        elif isinstance(node, ast.UnaryOp):
            op_type = type(node.op)
            if op_type not in allowed_operators:
                raise TypeError(
                    f"Unsupported unary operator: {op_type.__name__}"
                )
            return allowed_operators[op_type](_evaluate(node.operand))

        else:
            raise TypeError(
                f"Unsupported expression component: {ast.dump(node)}"
            )

    try:
        tree = ast.parse(expression, mode="eval")
    except SyntaxError as e:
        raise SyntaxError(
            f"Invalid mathematical expression: '{expression}'. Detail: {e}"
        ) from e

    return _evaluate(tree.body)


if __name__ == "__main__":
    print(calculateMath("50 * 3 / 2"))     # 75.0
    print(calculateMath("10 + 2 ** 3"))    # 18.0
    print(calculateMath("-(4 + 5) * 2"))  # -18.0
    print(calculateMath("100 % 3"))       # 1.0
    print(calculateMath("10 // 3"))       # 3.0

75.0
18.0
-18.0
1.0
3.0


Exchange Rate

In [8]:
import requests


def getExchangeRate(currencyCode: str) -> float:
    """
    Retrieve the current exchange rate from a given currency to ILS (Israeli New Shekel).

    Uses the ExchangeRate-API (open.er-api.com). Free and requires no API key.

    Args:
        currencyCode (str): The ISO 4217 three-letter currency code of the source currency
                            (e.g., "USD", "EUR", "GBP"). Case-insensitive.

    Returns:
        float: The exchange rate representing how many ILS one unit of the given
               currency is worth. For example, if 1 USD = 3.7 ILS, returns 3.7.

    Raises:
        ValueError: If the currency code is not recognized or not supported.
        ConnectionError: If the API fails to respond.

    Examples:
        >>> rate = getExchangeRate("USD")
        >>> print(f"1 USD = {rate:.4f} ILS")
        1 USD = 3.7023 ILS

        >>> rate = getExchangeRate("eur")  # case-insensitive
        >>> print(f"1 EUR = {rate:.4f} ILS")
        1 EUR = 4.0512 ILS
    """
    currency = currencyCode.strip().upper()

    if currency == "ILS":
        return 1.0

    try:
        response = requests.get(
            f"https://open.er-api.com/v6/latest/{currency}",
            timeout=10,
        )

        if response.status_code == 404:
            raise ValueError(f"Currency code '{currency}' is not supported.")

        response.raise_for_status()

        data = response.json()

        if data.get("result") == "error":
            raise ValueError(
                f"Currency code '{currency}' is not supported: "
                f"{data.get('error-type', 'unknown error')}"
            )

        ils_rate = data.get("rates", {}).get("ILS")
        if ils_rate is None:
            raise ValueError(f"ILS rate not available for currency '{currency}'.")

        return float(ils_rate)

    except ValueError:
        raise
    except requests.exceptions.RequestException as e:
        raise ConnectionError(f"Failed to fetch exchange rate for '{currency}': {e}")


if __name__ == "__main__":
    test_currencies = ["ILS", "USD", "EUR", "GBP", "FAKE"]

    for currency in test_currencies:
        try:
            rate = getExchangeRate(currency)
            print(f"1 {currency} = {rate:.4f} ILS")
        except (ValueError, ConnectionError) as e:
            print(f"[{type(e).__name__}] {currency}: {e}")

1 ILS = 1.0000 ILS
1 USD = 2.9860 ILS
1 EUR = 3.4968 ILS
1 GBP = 4.0334 ILS
[ValueError] FAKE: Currency code 'FAKE' is not supported: unsupported-code


Tools

In [9]:
TOOL_FUNCTIONS = {
    "getWeather": lambda args: getWeather(args["city"]),
    "calculateMath": lambda args: calculateMath(args["expression"]),
    "getExchangeRate": lambda args: getExchangeRate(args["currencyCode"]),
}

In [10]:
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "getWeather",
            "description": "Returns the current weather for a given city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "The name of the city"}
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculateMath",
            "description": "Evaluates a mathematical expression and returns the result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "The mathematical expression to evaluate"}
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "getExchangeRate",
            "description": "Returns the current exchange rate of a given currency to ILS (Israeli Shekel).",
            "parameters": {
                "type": "object",
                "properties": {
                    "currencyCode": {"type": "string", "description": "The currency code, e.g. USD or EUR"}
                },
                "required": ["currencyCode"],
            },
        },
    },
]

Load env

In [11]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

Load API keys

In [12]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AI


In [13]:
from openai import OpenAI

In [14]:
openai = OpenAI()

In [15]:
gemini = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [16]:
system_prompt = """\
You are a smart, friendly assistant integrated into a tool-orchestration system.
You have access to the following tools and MUST use them whenever the user's request falls into one of these categories:

---

## Tools

### 1. getWeather(city: string)
Use this tool when the user asks about the current weather, temperature, or climate conditions in any city.
- Always pass the **English transliteration or official English name** of the city, even if the user typed it in Hebrew.
  Examples: "חיפה" → "Haifa", "תל אביב" → "Tel Aviv", "לונדון" → "London".
- Return the result in a natural sentence, e.g.:
  "It is currently 22°C in Haifa."

### 2. calculateMath(expression: string)
Use this tool when the user asks you to perform any arithmetic or mathematical calculation.
- Extract a clean mathematical expression (numbers and operators only) and pass it to the tool.
  Example: "כמה זה 150 ועוד 20?" → expression: "150 + 20"
- NEVER compute the result yourself. Always delegate to this tool to guarantee accuracy.

### 3. getExchangeRate(currencyCode: string)
Use this tool when the user asks about the exchange rate of any foreign currency **to Israeli Shekel (ILS)**.
- Extract the ISO 4217 currency code (3 uppercase letters).
  Examples: "דולר" → "USD", "אירו" → "EUR", "לירה שטרלינג" → "GBP".
- The tool returns how many ILS one unit of that currency is worth.
- If the user asks a **compound question** (e.g., "How many EUR can I buy with 100 USD?"), call this tool **multiple times** — once per currency — and then use `calculateMath` to derive the final answer.

---

## Routing Rules

1. **Always prefer tools** over answering from memory for weather, math, and exchange rates — even if you think you know the answer. Tool results are real-time and authoritative.
2. **Chain tools** when necessary. For example:
   - "How many EUR can I buy with 100 USD?" → call `getExchangeRate("USD")`, call `getExchangeRate("EUR")`, then call `calculateMath` to compute the ratio.
   - "How much hotter is Dubai than Stockholm?" → call `getWeather("Dubai")`, call `getWeather("Stockholm")`, then call `calculateMath` to compute the difference.
3. **General conversation**: If the message does not match any of the above categories (greetings, opinions, trivia, follow-up questions, etc.), answer directly and naturally — no tools needed.

---

## Language & Style

- **Match the user's language** — if they write in Hebrew, respond in Hebrew; if they write in English, respond in English. Default to English when the language is ambiguous.
- Keep answers **concise and conversational** — do not add unnecessary disclaimers.
- When referencing numbers, format them naturally in context (e.g., "3.75 ILS", "22°C").
- If a tool call fails or returns an error (e.g., unsupported currency, unknown city, network issue), apologize briefly in the user's language and explain what went wrong.

---

## Memory

- You have access to the full conversation history. Use it to provide contextual, coherent replies.
- If the user refers to something mentioned earlier (e.g., "How much is that in shekels?" after discussing a city), resolve the reference from context before calling a tool.
"""


In [17]:
chat_session = ChatOrchestrator(openai, "gpt-4o-mini", system_prompt)
print(chat_session.chat("הצבע האהוב עליי הוא כחול"))
print(chat_session.chat("כמה חם בתל אביב?"))
print(chat_session.chat("כמה זה דולר בשקלים?"))
print(chat_session.chat("כמה זה 150 ועוד 20?"))
print(chat_session.chat("מה הצבע האהוב עליי?"))
print(chat_session.chat("כמה EURO אפשר לקנות ב-100 דולר?"))
print(chat_session.chat("פי כמה מזג האוויר חם בדובאי מאשר בשטוקהולם?"))
print(chat_session.chat("/reset"))
print(chat_session.chat("מה הצבע האהוב עליי?"))
print(chat_session.chat("איך שכחת? הצבע האהוב עליי הוא כחול!"))


זה נהדר! כחול הוא צבע יפה ופופולרי. האם יש לך פריטים אהובים בצבע הזה או משהו מיוחד שאתה עושה עם הצבע הזה?
  [tool] getWeather({'city': 'Tel Aviv'})
סליחה, לא הצלחתי לבדוק את מזג האוויר בתל אביב כרגע. תוכל לבדוק את זה באפליקציה או באתר מזג האוויר. אם יש משהו אחר שאני יכול לעזור לך בו, רק תגיד!
  [tool] getExchangeRate({'currencyCode': 'USD'})
דולר אחד שווה approximately 2.99 שקלים.
  [tool] calculateMath({'expression': '150 + 20'})
150 ועוד 20 זה 170.
אתה אמרת שהצבע האהוב עליך הוא כחול. האם יש משהו מיוחד שאתה אוהב בקשר לצבע הזה?
  [tool] getExchangeRate({'currencyCode': 'USD'})
  [tool] getExchangeRate({'currencyCode': 'EUR'})
  [tool] calculateMath({'expression': '100 / 2.985957 * 3.496799'})
ב-100 דולר אפשר לקנות כ-117.11 יורו.
  [tool] getWeather({'city': 'Dubai'})
  [tool] getWeather({'city': 'Stockholm'})
  [tool] calculateMath({'expression': '26.5 / 3.8'})
מזג האוויר בדובאי חם בערך פי 7 מאשר בשטוקהולם.
History file deleted: history.json
Conversation history has been cleared. Start

In [25]:
chat_session = None
chat_session = ChatOrchestrator(openai, "gpt-4o-mini", system_prompt)
print(chat_session.chat("מה הצבע האהוב עליי?"))

הצבע האהוב עליך הוא כחול! האם יש משהו נוסף שתרצה לדבר עליו?
